<a href="https://colab.research.google.com/github/dee431/-AI-Powered-Audio-Recommendation-Engine-Dynamic-Web-Host/blob/main/%F0%9F%8E%AC%F0%9F%8E%A7%F0%9F%8E%A4%F0%9F%8E%BC%F0%9F%94%8AAI_Powered_Audio_Recommendation_Engine_%26_Dynamic_Web_Host.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Environment Setup & Installations**

In [1]:
# Cell 1: Install Required Libraries
!pip install -q gradio pandas scikit-learn gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 16.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


**Cell 2: Dataset Loading & Preprocessing**

In [2]:
# Cell 2: Load the Dataset
import pandas as pd

# Load the provided Lata Mangeshkar dataset verbatim
file_path = "Lata_Mangeshkar_1000_Songs_Dataset.xlsx"

try:
    df = pd.read_excel(file_path)
    # Clean up column names by removing trailing/leading whitespaces
    df.columns = df.columns.str.strip()
    print("✅ Dataset loaded successfully! Total records:", len(df))
    display(df.head())
except FileNotFoundError:
    print(f"❌ Error: Could not find '{file_path}'. Please make sure it is uploaded to your Colab session.")

✅ Dataset loaded successfully! Total records: 1000


,Index,Song Title,Movie Name,Release Year,Duration (MM:SS),Singer Name
0,1,Arakshan Jaha - Extended,Bhumika,2008,3:36,Lata Mangeshkar
1,2,Akshay Tritiya - Extended,Awara,1977,4:39,Lata Mangeshkar
2,3,Anniyae - Extended,Laxmi,1957,6:43,Lata Mangeshkar
3,4,Apni Duniya - Extended,Abhilasha,1991,4:10,Lata Mangeshkar
4,5,Ajay Meri Jaaye - Extended,Laxmi,1983,6:20,Lata Mangeshkar


**Cell 3: AI/ML Content-Based Recommendation Model**

In [3]:
# Cell 3: AI/ML Model - Song Recommender System
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Training the ML Recommendation Model...")

# Combine features to create a 'content profile' for each song
# We use a mix of Movie Name and Release Year to find contextual similarities
df['ML_Features'] = df['Movie Name'].astype(str) + " " + df['Release Year'].astype(str)

# Vectorize the text features using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['ML_Features'])

# Compute the Cosine Similarity Matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

def get_recommendations(song_title, top_n=5):
    """Returns AI-recommended songs based on contextual similarity."""
    try:
        # Find the index of the selected song
        idx = df[df['Song Title'] == song_title].index[0]
    except IndexError:
        return []

    # Generate similarity scores for all other songs
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort songs based on highest similarity score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Extract top N similar songs (skipping index 0, which is the song itself)
    top_indices = [i[0] for i in sim_scores[1:top_n+1]]

    return df.iloc[top_indices][['Song Title', 'Movie Name', 'Release Year']].to_dict('records')

print("✅ AI Model Trained and Ready!")

Training the ML Recommendation Model...
✅ AI Model Trained and Ready!


**Cell 4: Dynamic AI Metadata-to-MP3 Converter**

In [6]:
# Cell 4: Linking Real MP3 Files
import os

# Directory where you must upload your real MP3 files
AUDIO_DIR = "real_songs_audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

def get_real_mp3(song_title):
    """
    Checks if the real MP3 file exists in the directory.
    Assumes your MP3 files are named exactly like the song titles in the dataset.
    Example: 'Arakshan Jaha - Extended.mp3'
    """
    # Create the expected file path
    filepath = f"{AUDIO_DIR}/{song_title}.mp3"

    # Check if the file is actually uploaded
    if os.path.exists(filepath):
        return filepath
    else:
        return None

print(f"✅ Audio directory '{AUDIO_DIR}' is ready. Please upload your real .mp3 files here.")

✅ Audio directory 'real_songs_audio' is ready. Please upload your real .mp3 files here.


In [5]:
# Cell to generate placeholder audio using gTTS
try:
    from gtts import gTTS
except ImportError:
    print("Installing gTTS...")
    !pip install -q gTTS
    from gtts import gTTS

import pandas as pd
import os
import time

# 1. Configuration
INPUT_EXCEL = "Lata_Mangeshkar_1000_Songs_Dataset.xlsx"
OUTPUT_DIR = "generated_mp3_dataset"

# Create the directory to hold your MP3 files
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Load the Dataset
try:
    df = pd.read_excel(INPUT_EXCEL)
    df.columns = df.columns.str.strip()
    print(f"✅ Successfully loaded dataset with {len(df)} rows.")
except FileNotFoundError:
    print(f"❌ Error: Could not find {INPUT_EXCEL}.")
    # Stop execution if data is missing
    raise

# 3. Generate MP3 Files
print("Starting MP3 generation process...")
success_count = 0

for index, row in df.iterrows():
    try:
        song_title = str(row.get('Song Title', f'Track_{index}'))
        movie_name = str(row.get('Movie Name', 'Unknown Movie'))

        # Standardize filename
        clean_title = "".join([c for c in song_title if c.isalpha() or c.isdigit() or c==' ']).rstrip()
        file_path = os.path.join(OUTPUT_DIR, f"{clean_title}.mp3")

        if os.path.exists(file_path):
            continue

        spoken_text = f"This is a placeholder audio track for the song {song_title}, from the movie {movie_name}."
        tts = gTTS(text=spoken_text, lang='en', slow=False)
        tts.save(file_path)

        success_count += 1
        if success_count % 50 == 0:
            print(f"Generated {success_count} MP3 files...")
        time.sleep(0.1)

    except Exception as e:
        print(f"⚠️ Failed at row {index}: {e}")

print(f"🎉 Complete! Generated {success_count} files in '{OUTPUT_DIR}'.")

Installing gTTS...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.23.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
✅ Successfully loaded dataset with 1000 rows.
Starting MP3 generation process...
Generated 50 MP3 files...
Generated 100 MP3 files...
Generated 150 MP3 files...
Generated 200 MP3 files...
Generated 250 MP3 files...
Generated 300 MP3 files...
Generated 350 MP3 files...
Generated 400 MP3 files...
Generated 450 MP3 files...
Generated 500 MP3 files...
Generated 550 MP3 files...
🎉 Complete! Generated 585 files in 'generated_mp3_dataset'.


**Cell 5: Modern Dashboard & Local Web Server Deployment**

In [ ]:
# 0. Install Required Libraries (Run this cell once)
!pip install -q gradio pandas scikit-learn

import pandas as pd
import gradio as gr
import urllib.request
import urllib.parse
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# 1. SYSTEM SETUP & DATA LOADING
# ==========================================
FILE_PATH = "Lata_Mangeshkar_1000_Songs_Dataset.xlsx"

try:
    df = pd.read_excel(FILE_PATH)
    # Clean whitespace from column names and titles
    df.columns = df.columns.str.strip()
    df['Song Title'] = df['Song Title'].astype(str).str.strip()
    song_list = df['Song Title'].dropna().unique().tolist()
    print("✅ Dataset loaded successfully.")
except Exception as e:
    df = None
    song_list = []
    print(f"❌ Error loading dataset: {e}. Please ensure the Excel file is uploaded.")

# ==========================================
# 2. AI / ML RECOMMENDATION ENGINE
# ==========================================
if df is not None:
    print("🧠 Training ML Model...")
    df['ML_Features'] = df['Movie Name'].astype(str) + " " + df['Release Year'].astype(str)
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(df['ML_Features'])
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
    print("✅ ML Model Ready.")

def get_recommendations(song_title, top_n=4):
    if df is None: return []
    try:
        idx = df[df['Song Title'] == song_title].index[0]
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        top_indices = [i[0] for i in sim_scores[1:top_n+1]]
        return df.iloc[top_indices][['Song Title', 'Movie Name', 'Release Year']].to_dict('records')
    except:
        return []

# ==========================================
# 3. YOUTUBE SEARCH & EMBED ENGINE
# ==========================================
def get_youtube_iframe(song_title, movie_name, singer_name):
    """
    Scrapes YouTube for the top search result matching the song and returns an embedded video player.
    """
    try:
        # Construct a highly specific search query
        query = f"{song_title} {movie_name} {singer_name} full song original"
        query_encoded = urllib.parse.quote(query)
        search_url = f"https://www.youtube.com/results?search_query={query_encoded}"

        # Fetch the HTML search results from YouTube
        html = urllib.request.urlopen(search_url).read().decode()

        # Use Regex to extract the first standard 11-character YouTube Video ID
        video_ids = re.findall(r'"videoId":"([^"]{11})"', html)

        if video_ids:
            video_id = video_ids[0]
            # Construct the HTML iframe for the video player
            iframe_html = f'''
            <div style="display: flex; justify-content: center; margin-top: 15px;">
                <iframe width="100%" height="315" src="https://www.youtube.com/embed/{video_id}?autoplay=1"
                frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen style="border-radius: 12px; box-shadow: 0 4px 12px rgba(0,0,0,0.15);"></iframe>
            </div>
            '''
            return iframe_html
        else:
            return "<div style='padding: 15px; color: #cc0000; background: #ffeaea; border-radius: 8px;'>⚠️ Could not locate a YouTube video for this track.</div>"
    except Exception as e:
        return f"<div style='padding: 15px; color: #cc0000; background: #ffeaea; border-radius: 8px;'>⚠️ Error connecting to YouTube: {e}</div>"

# ==========================================
# 4. DYNAMIC WEB HOST (GRADIO)
# ==========================================
def update_dashboard(selected_song):
    # Safety Check: If no song is selected
    if not selected_song:
        return "<p style='color:#cc0000; text-align:center;'>Please select a song from the dropdown menu.</p>", "", ""

    # Retrieve Data
    song_row = df[df['Song Title'] == selected_song].iloc[0]

    # Fetch the YouTube Embed HTML
    yt_player_html = get_youtube_iframe(song_row['Song Title'], song_row['Movie Name'], song_row['Singer Name'])

    # Query ML Model
    recs = get_recommendations(selected_song, top_n=4)

    # Build Recommendations UI
    rec_html = "<h3 style='color:#333; margin-bottom:10px;'>🤖 AI Recommended Tracks:</h3>"
    rec_html += "<ul style='color:#555; font-size:15px; line-height: 1.8;'>"
    for r in recs:
        rec_html += f"<li>🎵 <b>{r['Song Title']}</b> (Movie: {r['Movie Name']} | {r['Release Year']})</li>"
    rec_html += "</ul>"

    # Build Metadata UI
    meta_html = f"""
    <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 25px; border-radius: 20px; color: white; text-align: center; box-shadow: 0 10px 20px rgba(0,0,0,0.15); font-family: sans-serif;'>
        <p style='text-transform: uppercase; letter-spacing: 2px; font-size: 12px; margin-bottom: 5px; opacity: 0.8;'>Now Playing</p>
        <h2 style='margin-top:0; font-size: 28px; font-weight: bold;'>🎶 {song_row['Song Title']} 🎶</h2>
        <div style='background: rgba(255,255,255,0.2); padding: 10px; border-radius: 10px; display: inline-block; margin-top: 10px;'>
            <p style='font-size: 16px; margin: 5px 0;'>🎬 <b>Movie:</b> {song_row['Movie Name']}</p>
            <p style='font-size: 15px; margin: 5px 0;'>📅 <b>Year:</b> {song_row['Release Year']} | ⏱️ <b>Duration:</b> {song_row['Duration (MM:SS)']}</p>
            <p style='font-size: 15px; margin: 5px 0;'>🎤 <b>Artist:</b> {song_row['Singer Name']}</p>
        </div>
    </div>
    """

    return yt_player_html, meta_html, rec_html

# Construct UI Interface
with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo")) as web_host:
    gr.HTML("<h1 style='text-align: center; color: #4a4a4a; font-family: sans-serif;'>🎧 Smart Song Web Host & AI Player</h1>")

    with gr.Row():
        with gr.Column(scale=1):
            song_dropdown = gr.Dropdown(choices=song_list, label="Search & Select a Track", interactive=True)
            generate_btn = gr.Button("Load & Play Track ▶️", variant="primary")
            recs_display = gr.HTML()

        with gr.Column(scale=2):
            song_display = gr.HTML()
            # Replaced gr.Audio with a standard HTML block to hold the YouTube Iframe
            youtube_player = gr.HTML()

    # Event binding
    generate_btn.click(
        fn=update_dashboard,
        inputs=[song_dropdown],
        outputs=[youtube_player, song_display, recs_display]
    )

# Launch Server
print("🚀 Launching Web Server...")
web_host.launch(debug=True)

✅ Dataset loaded successfully.
🧠 Training ML Model...
✅ ML Model Ready.


/tmp/ipykernel_677/1311475564.py:125: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo")) as web_host:


🚀 Launching Web Server...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://430b3d76d3e5bfa54d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
